In [8]:
import pandas as pd

df = pd.read_csv("train/data.csv")
df.rename(columns={'index': 'filename'}, inplace=True)
print(df)


     filename  label
0       K0000      0
1       K0001      0
2       K0002      1
3       K0003      0
4       K0004      1
...       ...    ...
1612    K1612      1
1613    K1613      1
1614    K1614      0
1615    K1615      0
1616    K1616      0

[1617 rows x 2 columns]


In [9]:
import torch
print(torch.__version__)
print("MPS available:", torch.backends.mps.is_available())


2.9.0
MPS available: True


**Data Preparation**

We nornalize the pixedls here because pretrained CNNs expect pixel values that match ImageNet's distribution

In [10]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224 , 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor() ,
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
])

In [19]:
from torchvision.io import read_image
from torch.utils.data import Dataset , DataLoader
from PIL import Image
import os

class KilnDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.labels = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_id = self.labels.iloc[idx, 0]
        label = float(self.labels.iloc[idx, 1])
        img_path = os.path.join(self.img_dir, f"{img_id}.png")
        image = Image.open(img_path).convert("RGB")  # keep all 3 channels
        if self.transform:
            image = self.transform(image)
        return image, label

In [20]:
from torch.utils.data import random_split

train_dataset = KilnDataset(csv_file = "train/data.csv" , img_dir = "train" , transform = transform)
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size
train_ds , test_ds = random_split(train_dataset , [train_size , val_size])

train_loader = DataLoader(train_ds , batch_size = 32 , shuffle = True)

val_loader = DataLoader(test_ds , batch_size = 32 , shuffle = False)


In [24]:
images , labels = next(iter(train_loader))
print(images.shape ,images.shape)

torch.Size([32, 3, 224, 224]) torch.Size([32, 3, 224, 224])


In [26]:
import torch 
import torch.nn as nn
import torchvision.models as models

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

model = models.resnet18(pretrained = True)
model.fc = nn.Linear(model.fc.in_features , 1)
model = model.to(device)

/Users/hazelzhao/miniforge3/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/hazelzhao/miniforge3/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/hazelzhao/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 56.5MB/s]


In [28]:
pip install torchmetrics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 18.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [29]:
from torchmetrics.classification import BinaryAUROC
from tqdm import tqdm

criterioin = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters() , lr = 1e-4)
metric = BinaryAUROC().to(device)


In [35]:
for epoch in range(5):
    model.train()
    running_loss = 0
    for X , y in tqdm(train_loader):
        X = X.to(device)
        y = y.to(torch.float32).unsqueeze(1).to(device)
        optimizer.zero_grad()
        preds = model(X)
        loss = criterioin(preds , y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        print(f"Epoch {epoch+1}, Train loss: {running_loss/len(train_loader):.4f}")

        model.eval()
        with torch.no_grad():
            all_preds , all_labels = [] , []
            for X , y in val_loader:
                X = X.to(device)
                y = y.to(torch.float32).unsqueeze(1).to(device)
                preds = torch.sigmoid(model(X))
                all_preds.append(preds)
                all_labels.append(y)
            all_preds = torch.cat(all_preds)
            all_labels = torch.cat(all_labels)
            # all_labels = torch.cat(all_lables)
            auc = metric(all_preds , all_labels)
            print(f"Validation AUC: {auc.item():.4f}") 
        

  0%|          | 0/41 [00:00<?, ?it/s]

Epoch 1, Train loss: 0.0159


  2%|▏         | 1/41 [00:09<06:39,  9.98s/it]

Validation AUC: 0.6022
Epoch 1, Train loss: 0.0309


  5%|▍         | 2/41 [00:12<03:29,  5.37s/it]

...

Validation AUC: 0.9251
Epoch 5, Train loss: 0.2853


 98%|█████████▊| 40/41 [01:17<00:01,  1.98s/it]

Validation AUC: 0.9205
Epoch 5, Train loss: 0.2899


100%|██████████| 41/41 [01:19<00:00,  1.94s/it]

Validation AUC: 0.9333


In [37]:
 
with torch.no_grad():
    predicted_labels = (all_preds > 0.5)
    accuracy = (predicted_labels.eq(all_labels)).float().mean().mul(100).item()

print(f"Validation Accuracy: {accuracy:.2f}%")


Validation Accuracy: 83.33%


In [40]:
torch.save(model.state_dict(), "resnet18_kiln.pt")


In [42]:
import glob
from PIL import Image
import torch

class KilnTestDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_paths = sorted(glob.glob(os.path.join(img_dir, "*.png")))
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        img_id = os.path.basename(img_path).replace(".png", "")
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, img_id

test_dataset = KilnTestDataset("test", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [45]:
predictions = []
with torch.no_grad():
    for X, img_ids in tqdm(test_loader):
        X = X.to(device)
        preds = torch.sigmoid(model(X)).squeeze(1).cpu()
        for img_id, score in zip(img_ids, preds):
            predictions.append((img_id, score.item()))

# --- Step 5: save to CSV ---
submission = pd.DataFrame(predictions, columns=["index", "score"])
submission.to_csv("submission.csv", index=False)

100%|██████████| 23/23 [00:05<00:00,  4.10it/s]


In [47]:
binary_preds = [(img_id, 1 if score >= 0.5 else 0) for img_id, score in predictions]

submission = pd.DataFrame(binary_preds, columns=["index", "score"])
submission.to_csv("submission_binary.csv", index=False)

print("✅ Binary submission file created: submission_binary.csv")
print(submission.head())

✅ Binary submission file created: submission_binary.csv
   index  score
0  K1617      1
1  K1618      1
2  K1619      1
3  K1620      1
4  K1621      1


In [48]:
import os
train_files = set(os.listdir("train"))
test_files = set(os.listdir("test"))
print(len(train_files & test_files))


0
